# Livingston Township (07039) — Sales Trends

Historical sales-trend analysis from county property records (`data/merged.csv`).

Pipeline: **load & clean** → **filter to a segment** → **plot trends** (median price, volume, $/sqft).

Edit the `FILTERS` cell to slice by year range, beds, or property class, then re-run the plot cells.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.float_format", lambda v: f"{v:,.0f}")
plt.rcParams["figure.figsize"] = (11, 4.5)

DATA = Path("..") / "data" / "merged.csv"
assert DATA.exists(), f"missing {DATA.resolve()} — run the merge step first"

## 1. Load & clean

Parses dates and numerics, then flags rows that should be excluded from price trends:

- **non-arms-length / nominal sales** — price < $1,000 (the data has thousands of $0–$10 transfers: estate, LLC, intra-family, deed corrections).
- **unparseable sale dates**.

`df` = everything; `clean` = arms-length residential-usable rows. `$/sqft` additionally needs `Sq Ft > 0`.

In [ ]:
raw = pd.read_csv(DATA, dtype=str)

df = raw.copy()
df["sale_date"] = pd.to_datetime(df["Sale Date"], format="%m/%d/%Y", errors="coerce")
df["price"]   = pd.to_numeric(df["Sale Price"], errors="coerce")
df["sqft"]    = pd.to_numeric(df["Sq Ft"], errors="coerce")
df["beds"]    = pd.to_numeric(df["Beds"], errors="coerce")
df["year"]    = df["sale_date"].dt.year
df["ppsf"]    = df["price"] / df["sqft"].where(df["sqft"] > 0)

MIN_PRICE = 1_000
df["arms_length"] = df["price"] >= MIN_PRICE

clean = df[df["sale_date"].notna() & df["arms_length"]].copy()

print(f"raw rows:        {len(df):,}")
print(f"dropped (no date): {df['sale_date'].isna().sum():,}")
print(f"dropped (price < ${MIN_PRICE:,}): {(~df['arms_length']).sum():,}")
print(f"clean rows:      {len(clean):,}")
print(f"date range:      {clean['sale_date'].min():%Y-%m-%d} → {clean['sale_date'].max():%Y-%m-%d}")

## 2. Filters

Set the segment to analyze. `None` means "no filter on that field". Re-run from here down after changing.

In [ ]:
FILTERS = dict(
    property_class="Residential",   # e.g. "Residential", "Commercial", or None for all
    year_min=2000,                  # or None
    year_max=None,
    beds=None,                      # exact bed count, e.g. 3, or None
)

def apply_filters(data, property_class=None, year_min=None, year_max=None, beds=None):
    d = data
    if property_class is not None:
        d = d[d["Property Class"] == property_class]
    if year_min is not None:
        d = d[d["year"] >= year_min]
    if year_max is not None:
        d = d[d["year"] <= year_max]
    if beds is not None:
        d = d[d["beds"] == beds]
    return d

seg = apply_filters(clean, **FILTERS)
label = ", ".join(f"{k}={v}" for k, v in FILTERS.items() if v is not None) or "all sales"
print(f"segment: {label}")
print(f"rows in segment: {len(seg):,}")

## 3. Yearly trend table

Median price is the headline (robust to the occasional multi-million-dollar outlier); mean shown alongside. `ppsf` excludes zero-sqft rows.

In [ ]:
yearly = seg.groupby("year").agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    mean_price=("price", "mean"),
    median_ppsf=("ppsf", "median"),
).round(0)
yearly

## 4. Price trend over time

In [ ]:
fig, ax = plt.subplots()
ax.plot(yearly.index, yearly["median_price"], marker="o", label="median")
ax.plot(yearly.index, yearly["mean_price"], marker=".", alpha=0.5, label="mean")
ax.set_title(f"Sale price by year — {label}")
ax.set_ylabel("price ($)")
ax.yaxis.set_major_formatter(lambda v, _: f"${v/1000:,.0f}k")
ax.grid(alpha=0.3); ax.legend()
plt.show()

## 5. Sales volume by year

In [ ]:
fig, ax = plt.subplots()
ax.bar(yearly.index, yearly["sales"], color="steelblue")
ax.set_title(f"Number of sales by year — {label}")
ax.set_ylabel("sales")
ax.grid(alpha=0.3, axis="y")
plt.show()

## 6. Price per square foot by year

In [ ]:
ppsf = yearly["median_ppsf"].dropna()
fig, ax = plt.subplots()
ax.plot(ppsf.index, ppsf.values, marker="o", color="seagreen")
ax.set_title(f"Median $/sqft by year — {label}")
ax.set_ylabel("$ / sqft")
ax.yaxis.set_major_formatter(lambda v, _: f"${v:,.0f}")
ax.grid(alpha=0.3)
plt.show()

## 7. Export the yearly summary

In [ ]:
out = Path("..") / "output"
out.mkdir(exist_ok=True)
slug = label.replace(", ", "_").replace("=", "-").replace(" ", "")
path = out / f"yearly_{slug}.csv"
yearly.to_csv(path)
print("wrote", path)